# Real-time Communication

In [1]:
import networkx as nx 
from dataXplorer import JupyterClient
from dataXplorer import AnalysisToolkit, SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

In [2]:
# run backend (server) using buildandrun powershell script
client = JupyterClient()

#client.disconnect()

✅ Connected to /main


In [61]:
# get current directory
import os 
current_directory = os.getcwd()
print("Current Directory:", current_directory)

Current Directory: c:\Users\chris\Desktop\GIT\DataDiVR_WebApp


### Dataset "Social byzantine 14th century" : "Only people"

In [70]:
import networkx as nx
import xml.etree.ElementTree as ET

def create_people_multiplex_graph_from_file(file_path):
    """
    Transforms the social network data from an XML file into a NetworkX MultiGraph.
    Nodes are people, and multiple edges (relationships) are allowed between them.

    Args:
        file_path (str): The path to the XML file ('Only People.xml').

    Returns:
        networkx.MultiGraph: The constructed multiplex social network graph.
    """
    G = nx.Graph()
    people_data = {}    
    try:
        # Read content directly from the file path
        tree = ET.parse(file_path)
        root = tree.getroot()
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}. Please ensure the file is correctly provided.")
        return G
    except ET.ParseError as e:
        print(f"Error: Could not parse XML file at {file_path}. Error: {e}")
        return G

    # 1. First Pass: Find all nodes (people) and add them
    # The structure is DynamicMetaNetwork > MetaNetwork > nodes > nodeclass > node
    nodes_section = root.find('.//nodes/nodeclass[@type="Agent"]')
    if nodes_section is None:
        print("No Agent nodes found in the XML structure.")
        return G

    for node_elem in nodes_section.findall('node'):
        person_id = node_elem.get('id')
        if person_id:
            # Extract properties for each person
            person_name = person_id  # Use ID as default name
            properties = {}
            
            for prop in node_elem.findall('property'):
                prop_id = prop.get('id')
                prop_value = prop.get('value')
                if prop_id and prop_value:
                    properties[prop_id] = prop_value
                    
                    # Use 'Node Title' as name if available, otherwise keep ID
                    if prop_id == 'Node Title':
                        person_name = prop_value
            
            # Add node with all properties
            people_data[person_id] = {'id': person_id, 'name': person_name, **properties}
            G.add_node(person_id, label=person_name, **properties)

    # Parse links from all networks (keep network id/name as edge layer)
    for network in root.findall('.//networks/network'):
        layer = network.get('id') or network.get('name') or 'network'
        for link in network.findall('link'):
            s = link.get('source')
            t = link.get('target')
            if not s or not t:
                continue
            # collect link attributes (exclude source/target)
            edge_attrs = {k: v for k, v in link.attrib.items() if k not in ('source', 'target')}
            edge_attrs['layer'] = layer
            # ensure nodes exist (create minimal node if missing)
            if s not in G:
                G.add_node(s, label=s, type='unknown')
            if t not in G:
                G.add_node(t, label=t, type='unknown')
            # add or update edge attributes (Graph will keep one edge per node-pair)
            if G.has_edge(s, t):
                # merge attributes if edge already exists: prefer existing, but update with any new keys
                existing = G[s][t]
                merged = dict(existing)
                merged.update(edge_attrs)
                for key, value in merged.items():
                    G[s][t][key] = value
            else:
                G.add_edge(s, t, **edge_attrs)

    print("--- Graph Summary ---")
    print(f"Total Nodes: {G.number_of_nodes()}")
    print(f"Total Edges: {G.number_of_edges()}")
    print(f"Sample node properties: {list(people_data.keys())[:5] if people_data else 'None'}")
    
    return G

In [71]:
path_onlypeople = "temp-files/historynetworks/data/Social Network Multilayer Byzantine Elite 14th Century/Only People.xml"
people_multiplex_graph = create_people_multiplex_graph_from_file(path_onlypeople)

--- Graph Summary ---
Total Nodes: 856
Total Edges: 1117
Sample node properties: ['Ααρὼν Αλέξιος', 'Αβράμιος Μανουήλ', 'Αδριανός, Πέτρος Δούκας', 'Αγγελος Μανουήλ', '<Αγγελος>, Νικηφόρος ΙΙ. Δούκας']


In [72]:
# check all unique link attributes in graph 

unique_layers = set()
for u, v, attrs in people_multiplex_graph.edges(data=True):
    layer = attrs.get('layer', '')
    unique_layers.add(layer)

print("Unique link layers in people-multiplex graph:", unique_layers)


Unique link layers in people-multiplex graph: {'Friendship and Support'}


✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main


In [53]:
people_multiplex_graph.nodes(data=True)

NodeDataView({'Ααρὼν Αλέξιος': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Begin': '1393.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1393', 'Rank': 'Oikeios', 'PLP': '3.0'}, 'Αβράμιος Μανουήλ': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Begin': '1336.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1336', 'Rank': 'Doulos', 'PLP': '58.0'}, 'Αδριανός, Πέτρος Δούκας': {'label': 'Αδριανός, Πέτρος Δούκας', 'Function': 'Kurator d. Ασανίνα Φιλίππα in Thes/nike, 1349', 'Begin': '1349.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1349', 'Rank': 'Oikeios', 'PLP': '316.0'}, 'Αγγελος Μανουήλ': {'label': 'Αγγελος Μανουήλ', 'Function': 'ἐπὶ τοῦ κανικλείου, 1354 - 1370; καθολικὸς κριτής, 1352 - ca.1354', 'Begin': '1351.0', 'Loyalty': 'Prounionist', 'Secular/Ecclesiastical': 'Sec', 'End': '1370.0', 'PLP': '91040.0'}, '<Αγγελος>, Νικηφόρος ΙΙ. Δούκας': {'label': '<Αγγελος>, Νικηφόρος ΙΙ. Δούκας', 'Function': 'Graf von K

In [54]:
# links 
people_multiplex_graph.edges(data=True)

MultiEdgeDataView([('Ααρὼν Αλέξιος', 'Μιχαήλ, Erzbischof von Bethlehem', {}), ('Ααρὼν Αλέξιος', 'Μιχαήλ, Erzbischof von Bethlehem', {}), ('Αβράμιος Μανουήλ', 'Ασάνης Μιχαήλ', {}), ('Αβράμιος Μανουήλ', 'Ασάνης Μιχαήλ', {}), ('Αδριανός, Πέτρος Δούκας', 'Ασανίνα Φιλίππα', {}), ('Αδριανός, Πέτρος Δούκας', 'Ασανίνα Φιλίππα', {}), ('Αγγελος Μανουήλ', 'Γρηγορᾶς Νικηφόρος', {}), ('Αγγελος Μανουήλ', 'Γρηγορᾶς Νικηφόρος', {}), ('Αγγελος Μανουήλ', 'Καντακουζηνὴ Εἰρήνη, Kaiserin', {}), ('Αγγελος Μανουήλ', 'Καντακουζηνὴ Εἰρήνη, Kaiserin', {}), ('<Αγγελος>, Νικηφόρος ΙΙ. Δούκας', 'Ριτζάρδος', {}), ('<Αγγελος>, Νικηφόρος ΙΙ. Δούκας', 'Ριτζάρδος', {}), ('Αθανάσιος 422', 'Παλαιολόγος Ανδρόνικος ΙΙ.', {}), ('Αθανάσιος 422', 'Παλαιολόγος Ανδρόνικος ΙΙ.', {}), ('Ακροπολίτης Μελχισεδέκ', 'Φιλανθρωπηνός, Αλέξιος Ταρχανειώτης', {}), ('Ακροπολίτης Μελχισεδέκ', 'Φιλανθρωπηνός, Αλέξιος Ταρχανειώτης', {}), ('Ακροπολίτης Μελχισεδέκ', 'Πλανούδης Μανουὴλ', {}), ('Ακροπολίτης Μελχισεδέκ', 'Πλανούδης Μανουὴλ', {}), (

In [55]:
import cartoGRAPHs as cg
pos_3D_people = cg.layout_global_umap(people_multiplex_graph, dim=3, n_neighbors=10, min_dist=0.1)
pos_3D_people_ = {key: list(value) for key, value in pos_3D_people.items()}

c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\umap\umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [7]:
people_multiplex_graph.graph['projectname'] = "ByzNet-1400-people-only"
people_multiplex_graph.graph['info'] = "ByzNet-1400 visualises the multi-layered social, political, and kinship networks of the Late Byzantine elite " \
"(1282–1402 CE) using data derived from the PLP prosopography. The dataset connects 2402 individuals and 336 localities across ties of kinship, " \
"allegiance, friendship, conflict, diplomacy, and mobility, revealing how elite factions formed, fractured, " \
"and interacted during a period of civil war and imperial contraction. The project constructs a multiplex, temporal, " \
"and geospatial graph, enabling exploration of power clusters, conflict dynamics, family expansion, and the central role " \
"of Constantinople within a highly interconnected but increasingly polarised aristocratic world."

people_multiplex_graph.graph["layoutname"] ='00-global-cartographs'

nx.set_node_attributes(people_multiplex_graph, {node: data.copy() for node, data in people_multiplex_graph.nodes(data=True)}, name='annotation')
# delete all other attributes except 'annotation'
for node in people_multiplex_graph.nodes():
    annotation = people_multiplex_graph.nodes[node]['annotation']
    people_multiplex_graph.nodes[node].clear()
    people_multiplex_graph.nodes[node]['annotation'] = annotation

In [8]:
#!pip install romanize3

In [9]:
import romanize3

mapping = {}
# create mapping from node id to name
for node, data in people_multiplex_graph.nodes(data=True):
    name = data['annotation'].get('name', '')
    mapping[node] = name

# add roman letters name as attribute 
for node, data in people_multiplex_graph.nodes(data=True):
    name = node
    print(f"Original Name: {name}")

    roman_name = romanize3.__dict__['grc'].convert(name)
    print(f"Node: {node}, Transliterated: {roman_name}")

    data['annotation']['transliterated'] = roman_name

Original Name: Ααρὼν Αλέξιος
Node: Ααρὼν Αλέξιος, Transliterated: Aarὼn Alέcios
Original Name: Αβράμιος Μανουήλ
Node: Αβράμιος Μανουήλ, Transliterated: Abrάmios Manouήl
Original Name: Αδριανός, Πέτρος Δούκας
Node: Αδριανός, Πέτρος Δούκας, Transliterated: Adrianόs, Pέtros Doύkas
Original Name: Αγγελος Μανουήλ
Node: Αγγελος Μανουήλ, Transliterated: Aggelos Manouήl
Original Name: <Αγγελος>, Νικηφόρος ΙΙ. Δούκας
Node: <Αγγελος>, Νικηφόρος ΙΙ. Δούκας, Transliterated: <Aggelos>, Nikêfόros II. Doύkas
Original Name: Αθανάσιος 422
Node: Αθανάσιος 422, Transliterated: Ahanάsios 422
Original Name: Ακροπολίτης Μελχισεδέκ
Node: Ακροπολίτης Μελχισεδέκ, Transliterated: Akropolίtês Melxisedέk
Original Name: Αλουσιάνος Θωμᾶς <Δούκας>
Node: Αλουσιάνος Θωμᾶς <Δούκας>, Transliterated: Alousiάnos Hômᾶs <Doύkas>
Original Name: Ανδρονικόπουλος Ιωάννης
Node: Ανδρονικόπουλος Ιωάννης, Transliterated: Andronikόpoulos Iôάnnês
Original Name: Ανδρόνικος 952
Node: Ανδρόνικος 952, Transliterated: Andrόnikos 952
Origi

In [10]:
# check node information 
for node, data in list(people_multiplex_graph.nodes(data=True))[:5]:
    print(f"Node ID: {node}, Data: {data}")

Node ID: Ααρὼν Αλέξιος, Data: {'annotation': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Begin': '1393.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1393', 'Rank': 'Oikeios', 'PLP': '3.0', 'transliterated': 'Aarὼn Alέcios'}}
Node ID: Αβράμιος Μανουήλ, Data: {'annotation': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Begin': '1336.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1336', 'Rank': 'Doulos', 'PLP': '58.0', 'transliterated': 'Abrάmios Manouήl'}}
Node ID: Αδριανός, Πέτρος Δούκας, Data: {'annotation': {'label': 'Αδριανός, Πέτρος Δούκας', 'Function': 'Kurator d. Ασανίνα Φιλίππα in Thes/nike, 1349', 'Begin': '1349.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1349', 'Rank': 'Oikeios', 'PLP': '316.0', 'transliterated': 'Adrianόs, Pέtros Doύkas'}}
Node ID: Αγγελος Μανουήλ, Data: {'annotation': {'label': 'Αγγελος Μανουήλ', 'Function': 'ἐπὶ τοῦ κανικλείου, 1354 - 1370; καθολικὸς κριτής, 1352 - ca.1354', '

In [11]:
# color nodes by Secular/Ecclesiastical status
node_colors = {}
for node, data in people_multiplex_graph.nodes(data=True):
    status = data.get('annotation', {}).get('Secular/Ecclesiastical', 'Unknown')
    if status == 'Sec':
        node_colors[node] = (0,255,0,160)
    elif status == 'Eccl':
        node_colors[node] = (255,0,0,160)
    else:
        node_colors[node] = (128,128,128,100)  

In [12]:
nx.set_node_attributes(people_multiplex_graph, node_colors, name='nodecolor')
nx.set_edge_attributes(people_multiplex_graph, (80,80,80,100), name='linkcolor')
nx.set_node_attributes(people_multiplex_graph, pos_3D_people_, name='pos')

In [13]:
# Extract 'label' attribute for all nodes and set it as a new attribute 'name'
labels = {node: data.get('annotation', {}).get('label', 'Unknown') for node, data in people_multiplex_graph.nodes(data=True)}
nx.set_node_attributes(people_multiplex_graph, labels, name='name')

In [14]:
# check node information 
for node, data in list(people_multiplex_graph.nodes(data=True))[:5]:
    print(f"Node ID: {node}, Data: {data}")

Node ID: Ααρὼν Αλέξιος, Data: {'annotation': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Begin': '1393.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1393', 'Rank': 'Oikeios', 'PLP': '3.0', 'transliterated': 'Aarὼn Alέcios'}, 'nodecolor': (0, 255, 0, 160), 'pos': [0.0322755072, 0.703994189, 0.3016912717], 'name': 'Ααρὼν Αλέξιος'}
Node ID: Αβράμιος Μανουήλ, Data: {'annotation': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Begin': '1336.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1336', 'Rank': 'Doulos', 'PLP': '58.0', 'transliterated': 'Abrάmios Manouήl'}, 'nodecolor': (0, 255, 0, 160), 'pos': [0.2798681921, 0.871614149, 0.9096647999], 'name': 'Αβράμιος Μανουήλ'}
Node ID: Αδριανός, Πέτρος Δούκας, Data: {'annotation': {'label': 'Αδριανός, Πέτρος Δούκας', 'Function': 'Kurator d. Ασανίνα Φιλίππα in Thes/nike, 1349', 'Begin': '1349.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1349', 'Rank': 'Oikeios', 'PLP'

In [15]:
import networkx as nx 

# remap node ids to be numbers
people_multiplex_graph = nx.relabel_nodes(people_multiplex_graph, {old_id: new_id for new_id, old_id in enumerate(people_multiplex_graph.nodes())})

print(list(people_multiplex_graph.nodes(data=True))[:10])

[(0, {'annotation': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Begin': '1393.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1393', 'Rank': 'Oikeios', 'PLP': '3.0', 'transliterated': 'Aarὼn Alέcios'}, 'nodecolor': (0, 255, 0, 160), 'pos': [0.0322755072, 0.703994189, 0.3016912717], 'name': 'Ααρὼν Αλέξιος'}), (1, {'annotation': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Begin': '1336.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1336', 'Rank': 'Doulos', 'PLP': '58.0', 'transliterated': 'Abrάmios Manouήl'}, 'nodecolor': (0, 255, 0, 160), 'pos': [0.2798681921, 0.871614149, 0.9096647999], 'name': 'Αβράμιος Μανουήλ'}), (2, {'annotation': {'label': 'Αδριανός, Πέτρος Δούκας', 'Function': 'Kurator d. Ασανίνα Φιλίππα in Thes/nike, 1349', 'Begin': '1349.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1349', 'Rank': 'Oikeios', 'PLP': '316.0', 'transliterated': 'Adrianόs, Pέtros Doύkas'}, 'nodecolor': (0, 255, 0, 160)

In [16]:
import nx2json as nx2j 
nx2j.create_project(people_multiplex_graph)

Successfully created the directory static/projects/ByzNet-1400-people-only 
PROGRESS: loaded graph JSON...
PROGRESS: stored graph data...
PROGRESS: stored layouts...
PROGRESS: stored node info...
PROGRESS: made node position textures...
PROGRESS: made textures for node colors...
PROGRESS: made textures for links...
PROGRESS: made textures for linkcolors...
PROGRESS: writing json files for project and nodes...
Project created successfully.


### Access Project list and select one by ID

In [17]:
# see if new project is in projectlist
import GlobalData as GD

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, 'AE_Memes_2022'),
 (1, 'ARS23_memes'),
 (2, 'ByzNet-1400-people-and-locations'),
 (3, 'ByzNet-1400-people-only'),
 (4, 'CDK5'),
 (5, 'CircLadderGraph-xsmall'),
 (6, 'diffusion'),
 (7, 'JSON_autocore'),
 (8, 'JSON_barbellgraph'),
 (9, 'JSON_Zachary'),
 (10, 'Microplastics_HumanHealth'),
 (11, 'Pesticides_HumanHealth'),
 (12, 'PG_NEW'),
 (13, 'Powergrid_Europe'),
 (14, 'PPI_brain_infarction'),
 (15, 'PPI_joel_daniel_aryan'),
 (16, 'PPI_joel_daniel_aryan_C'),
 (17, 'PPI_joel_daniel_aryan_test_1'),
 (18, 'PPI_networkcartoGRAPHs'),
 (19, 'Realtime-project'),
 (20, 'Sphere_Torus'),
 (21, 'Teapot'),
 (22, 'Template'),
 (23, 'test'),
 (24, 'TheMandelbulb_edges'),
 (25, 'XX')]

In [18]:
# select a project to work with
sel_id = 3
sel_name = allprojects_updated[sel_id][1]

# load the project data
session = SessionManager(sel_id, sel_name, client)
session.load_graph_from_project()
session.reload_project()

# initialize 
file_mgr = ProjectFileManager(session)
tex_gen = TextureGenerator(session)
syncer = VisualizerSyncer(session)

Session Graph loaded from project folder. 
Project name:  ByzNet-1400-people-only
Data: Nodes: 856 Links: 1117


In [19]:
G = session.load_graph_from_project()

# get nodenames from node attributes
nodenames_session = [data.get("name") for _, data in G.nodes(data=True)]
nodenames_session = [name for name in nodenames_session if name is not None]

# rename nodes back to original names
mapping = dict(zip(G.nodes(), nodenames_session))
G = nx.relabel_nodes(G, mapping)
G.nodes()

Session Graph loaded from project folder. 
Project name:  ByzNet-1400-people-only
Data: Nodes: 856 Links: 1117


NodeView(('Ααρὼν Αλέξιος', 'Αβράμιος Μανουήλ', 'Αδριανός, Πέτρος Δούκας', 'Αγγελος Μανουήλ', '<Αγγελος>, Νικηφόρος ΙΙ. Δούκας', 'Αθανάσιος 422', 'Ακροπολίτης Μελχισεδέκ', 'Αλουσιάνος Θωμᾶς <Δούκας>', 'Ανδρονικόπουλος Ιωάννης', 'Ανδρόνικος 952', 'Ανδρόνικος 959', 'Αρτῶτος 1447', 'Ασάνης Ανδρέας', 'Ασάνης Ισαάκιος', 'Ασάνης Κωνσταντῖνος 1503', 'Ατταλειώτης', 'Βαρλαάμ', 'Βατάτζης Ιωάννης', 'Bούχειρ Ισίδωρος Ι.', 'Βράναινα Μαρία', 'Βρανᾶς Θεόδωρος', 'Βράνος', 'Βρυέννιος Ιωσήφ', 'Γαβαλᾶς Μιχαήλ', 'Γαβρᾶς Ιωάννης, Schriftsteller', 'Γαβρᾶς Μιχαήλ', 'Γαβριήλ, Mönch', 'Γάρσιας', 'Γελίελμος', 'Γεράσιμος 3747', 'Γερβάσιος 3792', 'Γλυκύς', 'Γουδέλης Γεώργιος', 'Γρηγορᾶς Νικηφόρος', 'Γρηγόριος, Erzbischof von Ochrid', 'Γρηγόριος, Mönchsvater', 'Δανιήλ, Metropolit von Ainos', 'Δεξιὸς Θεόδωρος', 'Δεσίσθλαβος', 'Δημήτριος, Diener', 'Διαβολάγγελος', 'Δισύπατος Γεώργιος', 'Δισύπατος Δαβίδ', 'Δουκαΐτης', 'Δουκόπουλος Πέτρος', 'Δρομορᾶς', 'Εἰρήνη 5972', 'Εξώτροχος Ανδρόνικος', 'Εσκαμματισμένος Λέων', 'Εὐρ

In [20]:
mapping

{0: 'Ααρὼν Αλέξιος',
 1: 'Αβράμιος Μανουήλ',
 2: 'Αδριανός, Πέτρος Δούκας',
 3: 'Αγγελος Μανουήλ',
 4: '<Αγγελος>, Νικηφόρος ΙΙ. Δούκας',
 5: 'Αθανάσιος 422',
 6: 'Ακροπολίτης Μελχισεδέκ',
 7: 'Αλουσιάνος Θωμᾶς <Δούκας>',
 8: 'Ανδρονικόπουλος Ιωάννης',
 9: 'Ανδρόνικος 952',
 10: 'Ανδρόνικος 959',
 11: 'Αρτῶτος 1447',
 12: 'Ασάνης Ανδρέας',
 13: 'Ασάνης Ισαάκιος',
 14: 'Ασάνης Κωνσταντῖνος 1503',
 15: 'Ατταλειώτης',
 16: 'Βαρλαάμ',
 17: 'Βατάτζης Ιωάννης',
 18: 'Bούχειρ Ισίδωρος Ι.',
 19: 'Βράναινα Μαρία',
 20: 'Βρανᾶς Θεόδωρος',
 21: 'Βράνος',
 22: 'Βρυέννιος Ιωσήφ',
 23: 'Γαβαλᾶς Μιχαήλ',
 24: 'Γαβρᾶς Ιωάννης, Schriftsteller',
 25: 'Γαβρᾶς Μιχαήλ',
 26: 'Γαβριήλ, Mönch',
 27: 'Γάρσιας',
 28: 'Γελίελμος',
 29: 'Γεράσιμος 3747',
 30: 'Γερβάσιος 3792',
 31: 'Γλυκύς',
 32: 'Γουδέλης Γεώργιος',
 33: 'Γρηγορᾶς Νικηφόρος',
 34: 'Γρηγόριος, Erzbischof von Ochrid',
 35: 'Γρηγόριος, Mönchsvater',
 36: 'Δανιήλ, Metropolit von Ainos',
 37: 'Δεξιὸς Θεόδωρος',
 38: 'Δεσίσθλαβος',
 39: 'Δημήτριος, D

### Manipulate node colors

In [21]:
# initialize analysis toolkit
tools = AnalysisToolkit(session, tex_gen, syncer, file_mgr)

In [22]:
# get attributes from nodes.json

# read nodes.json file 
import json

curr_project = sel_name 
current_directory = os.path.join("static", "projects")

try:
    with open(os.path.join(current_directory, curr_project, 'nodes.json'), 'r', encoding='utf-8') as f:
        nodes_data = json.load(f)
        print(f"Loaded nodes.json with {len(nodes_data)} entries.")
except FileNotFoundError:
    print(f"Error: File not found at {os.path.join(curr_project, 'nodes.json')}. Please ensure the file exists.")
    nodes_data = {}

print("Sample node data from nodes.json:", list(nodes_data.items())[:2])


nodes_attr_rank = {}
for ix,node in enumerate(G.nodes()):
    print(node)
    node_info = next((item for item in nodes_data.get('nodes', []) if str(node) == str(item.get('n'))), {})
    rank = node_info.get('attrlist', {}).get('Rank', 'Unknown')
    nodes_attr_rank[ix] = rank

nodes_attr_function = {}
for ix, node in enumerate(G.nodes()):
    node_info = next((item for item in nodes_data.get('nodes', []) if str(node) == str(item.get('n'))), {})
    function = node_info.get('attrlist', {}).get('Function', 'Unknown')
    nodes_attr_function[ix] = function

print("Sample node ranks:", list(nodes_attr_rank.items())[:10])
print("Sample node functions:", list(nodes_attr_function.items())[:10])

Loaded nodes.json with 1 entries.
Sample node data from nodes.json: [('nodes', [{'id': 0, 'n': 'Ααρὼν Αλέξιος', 'attrlist': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Begin': '1393.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1393', 'Rank': 'Oikeios', 'PLP': '3.0', 'transliterated': 'Aarὼn Alέcios'}}, {'id': 1, 'n': 'Αβράμιος Μανουήλ', 'attrlist': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Begin': '1336.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1336', 'Rank': 'Doulos', 'PLP': '58.0', 'transliterated': 'Abrάmios Manouήl'}}, {'id': 2, 'n': 'Αδριανός, Πέτρος Δούκας', 'attrlist': {'label': 'Αδριανός, Πέτρος Δούκας', 'Function': 'Kurator d. Ασανίνα Φιλίππα in Thes/nike, 1349', 'Begin': '1349.0', 'Secular/Ecclesiastical': 'Sec', 'End': '1349', 'Rank': 'Oikeios', 'PLP': '316.0', 'transliterated': 'Adrianόs, Pέtros Doύkas'}}, {'id': 3, 'n': 'Αγγελος Μανουήλ', 'attrlist': {'label': 'Αγγελος Μανουήλ', 'Fu

In [26]:
# Create a color map for ranks
import matplotlib.cm as cm

rank_color_map = {}
colormap = cm.get_cmap('Set1')  # Use the Set1 colormap

unique_ranks = set(nodes_attr_rank.values())

# check occurence of each rank
rank_counts = {rank: list(nodes_attr_rank.values()).count(rank) for rank in unique_ranks}
# sort by occurence
rank_counts = dict(sorted(rank_counts.items(), key=lambda item: item[1], reverse=True))
print("Rank counts:", rank_counts)

# Assign each unique rank a unique color
unique_ranks = list(rank_counts.keys())  # Get all unique ranks
num_colors = colormap.N  # Total number of colors in the colormap

for i, rank in enumerate(unique_ranks):
    if rank == "Unknown" or rank_counts[rank] < 2:
        rank_color_map[rank] = (60, 60, 60, 60)  # Default gray color
    else:
        color_index = i % num_colors
        rgba = colormap(color_index / num_colors)
        rank_color_map[rank] = (
            int(rgba[0] * 255),  # Red
            int(rgba[1] * 255),  # Green
            int(rgba[2] * 255),  # Blue
            110  # Alpha (transparency)
        )

# Print the legend of ranks and their assigned colors
print("Rank Color Legend:")
for rank, color in rank_color_map.items():
    print(f"Rank: {rank}, Color: {color}")

# assign node colors based on rank
node_colors_rank = {}
for node, rank in nodes_attr_rank.items():
    node_colors_rank[node] = rank_color_map.get(rank, (60,60,60,60))  # default gray

# create a color bitmap
tex_gen.generate_node_color_texture(
    node_colors_rank,
    texture_name="z_by_Rank",
    save=True
)

Rank counts: {'Unknown': 692, 'Oikeios': 49, 'Doulos': 19, 'Monachos': 5, 'Hieromonachos': 5, 'Sebastos': 4, 'Patriarch': 3, 'Kyr': 3, 'Eunuch': 3, 'Abt': 3, 'Archidiakon': 2, 'Panhypersebastos': 2, 'Pneumatikos': 2, 'Metropolit': 2, 'Pansebastos Sebastos': 2, 'Despot': 2, 'Geboren in Kpl als Sohn eines Türken und einer Griechin': 1, 'Diakon': 1, 'Priest': 1, 'Kaisarissa': 1, 'Mönch d. Laura-Kl.': 1, 'Ι., Despot, empfohlen worden, was er aber nicht verdiente.': 1, 'Konvertierter Jude': 1, 'Der mittellose Thronprätendent Σφεντίσθλαβος (Theodor Svetoslav, Zar von': 1, 'Abt d. Diomedes-Kl. in Kpl, 1374': 1, 'Paganino Doria, de Auria': 1, 'War früher Moslem. Καντακουζηνὸς Ιωάννης VI. unterwies ihn': 1, 'Lief wahrscheinlich 1352 zu Παλαιολόγος': 1, 'πανευγενέστατος': 1, 'von Stagoi/Thessalien, 1362; Hieromonachos, 1362 - 1372; Abt d. Dupiane-Kl': 1, 'Verwaltete vor seiner Amtserhebung zum Metropoliten das κελλίον': 1, 'Protobestiarios': 1, 'Abt d. Pammakaristos-Kl. in Kpl, zw. 1282': 1, 'Se

C:\Users\chris\AppData\Local\Temp\ipykernel_35572\2106877475.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colormap = cm.get_cmap('Set1')  # Use the Set1 colormap


'static/projects/ByzNet-1400-people-only\\layoutsRGB\\z_by_Rank.png'

In [33]:
# node colors by FUNCTION
import matplotlib.cm as cm

node_colors_function = {}
function_color_map = {}
colormap = cm.get_cmap('Set1')  # Use the Set1 colormap

unique_functions = set(nodes_attr_function.values())

# Check occurrence of each function
function_counts = {function: list(nodes_attr_function.values()).count(function) for function in unique_functions}
# Sort by occurrence
function_counts = dict(sorted(function_counts.items(), key=lambda item: item[1], reverse=True))
print("Function counts:", function_counts)

# Assign each unique function a unique color
unique_functions_sorted = list(function_counts.keys())  # Get all unique functions
num_colors = colormap.N  # Total number of colors in the colormap

for i, function in enumerate(unique_functions_sorted):
    if function == "Unknown" or function_counts[function] < 1:
        function_color_map[function] = (60, 60, 60, 60)  # Default gray color
    else:
        color_index = i % num_colors
        rgba = colormap(color_index / num_colors)
        function_color_map[function] = (
            int(rgba[0] * 255),  # Red
            int(rgba[1] * 255),  # Green
            int(rgba[2] * 255),  # Blue
            150  # Alpha (transparency)
        )

# Print the legend of functions and their assigned colors
print("Function Color Legend:")
for function, color in function_color_map.items():
    print(f"Function: {function}, Color: {color}")

# Assign node colors based on function
for node, function in nodes_attr_function.items():
    node_colors_function[node] = function_color_map.get(function, (60, 60, 60, 60))  # Default gray

# Create a color bitmap
tex_gen.generate_node_color_texture(
    node_colors_function,
    texture_name="z_by_Function",
    save=True
)


Function counts: {'Unknown': 315, 'Gesandter': 5, 'Mauerwächter in Kpl, 1328, Zimmermann': 2, 'Patriarch von Kpl.': 2, 'Protosebastos von Trapezunt, 1363': 1, 'Metropolit von Adrianopel, zw. 1278-11 - A. 1383, Schriftsteller; Mönch, vor': 1, 'Kaiser, 1425-07-21 - 1448-10-31; Mitkaiser, vor 1407 (seit 1403?)': 1, 'Dauphin von Vienne/Gascogne, 1333 - 1349': 1, 'Gefolgsmann d. Παλαιολόγος Ιωάννης VII., 1403': 1, 'Metropolit von Ephesos, 1278 - 1283': 1, 'Priester in Kpl, 1347, Lehrer': 1, 'Mitkaiserin, 1296 - 1320; Nonne, 1320 - 1333': 1, 'Priester in Kpl, 1390 - 1400; Hausbesitzer in Kpl, 1400': 1, 'Melograph, 1358 - 1415, Schriftsteller': 1, 'Nomophylax d. kaiserl. Klerus, 1344 - 1345; Sakelliu von Ochrid, ca. 1344': 1, 'Patriarch von Jerusalem, 1347-05/08 bzw. 2.H. 1349 - 1368': 1, 'Seefahrer aus Ancona, 1370 - 1372 (mindestens), Kaufmann': 1, 'Dominikaner, 1261 od. früher - ca. 1325, Schriftsteller': 1, 'Steuerbeamter (Apographeus, ἐξισωτής) bei Kpl, 1319/20 - 1342, Pansebastos': 1, '

C:\Users\chris\AppData\Local\Temp\ipykernel_35572\2420413141.py:6: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colormap = cm.get_cmap('Set1')  # Use the Set1 colormap


'static/projects/ByzNet-1400-people-only\\layoutsRGB\\z_by_Function.png'

In [34]:
# color by importance (degree, betweenness, etc.)

nodecolor_degree = {}
import matplotlib.cm as cm

# Use colormap YlOrRd
colormap = cm.get_cmap('YlOrRd')

degree_dict = dict(people_multiplex_graph.degree())
max_degree = max(degree_dict.values())
for node, degree in degree_dict.items():
    intensity = degree / max_degree
    rgba = colormap(intensity)
    nodecolor_degree[node] = (int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255), 110)

tex_gen.generate_node_color_texture(
    nodecolor_degree,
    texture_name="z_by_Degree",
    save=True       
)

C:\Users\chris\AppData\Local\Temp\ipykernel_35572\1451644165.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colormap = cm.get_cmap('YlOrRd')


'static/projects/ByzNet-1400-people-only\\layoutsRGB\\z_by_Degree.png'

In [35]:
# colors by closeness centrality
nodecolor_closeness = {}
closeness_dict = nx.closeness_centrality(people_multiplex_graph)
max_closeness = max(closeness_dict.values())
for node, closeness in closeness_dict.items():
    intensity = closeness / max_closeness
    rgba = colormap(intensity)
    nodecolor_closeness[node] = (int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255), 110)

tex_gen.generate_node_color_texture(
    nodecolor_closeness,
    texture_name="z_by_Closeness",
    save=True       
)

'static/projects/ByzNet-1400-people-only\\layoutsRGB\\z_by_Closeness.png'

In [36]:
# nodecolor by betweenness centrality
nodecolor_betweenness = {}
betweenness_dict = nx.betweenness_centrality(people_multiplex_graph)
max_betweenness = max(betweenness_dict.values())
for node, betweenness in betweenness_dict.items():
    intensity = betweenness / max_betweenness
    rgba = colormap(intensity)
    nodecolor_betweenness[node] = (int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255), 110)

tex_gen.generate_node_color_texture(
    nodecolor_betweenness,
    texture_name="z_by_Betweenness",
    save=True       
)

'static/projects/ByzNet-1400-people-only\\layoutsRGB\\z_by_Betweenness.png'

In [37]:
# make importance layout 

# transform multi graph to simple graph for layout
simple_graph = nx.Graph()
simple_graph.add_nodes_from(people_multiplex_graph.nodes(data=True))
simple_graph.add_edges_from(people_multiplex_graph.edges(data=True))

posG_3D_imp = cg.layout_importance_umap(simple_graph, dim=3,n_neighbors=10, min_dist=0.9)

# make node texture for importance layout
tex_gen.generate_node_position_texture(
    posG_3D_imp,
    texture_name="z_ImportanceLayout_3D",
    save=True
)


c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\umap\umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


('static/projects/ByzNet-1400-people-only/layouts/z_ImportanceLayout_3D.bmp',
 'static/projects/ByzNet-1400-people-only/layoutsl/z_ImportanceLayout_3Dl.bmp')

In [38]:
# make local layout
posG_3D_spring = nx.spring_layout(simple_graph, dim=3, seed=42, iterations=100)

# make node texture for importance layout
tex_gen.generate_node_position_texture(
    posG_3D_spring,
    texture_name="z_SpringLayout_3D",
    save=True
)


✅ Connected to /main
✅ Connected to /main
✅ Connected to /main


('static/projects/ByzNet-1400-people-only/layouts/z_SpringLayout_3D.bmp',
 'static/projects/ByzNet-1400-people-only/layoutsl/z_SpringLayout_3Dl.bmp')

✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
